# Chapter 12 — Let Old Context Fade

## Question

**Must information jump directly from full fidelity to one compact summary?**

Falsifiable structure: if retention class stays fixed while the active representation changes with pressure, retention and fidelity are separate variables. All representations below are deterministic fixtures; selection among them calls no model.

## Setup — episodes with legal representation ranges

Five episodes in the spirit of Chapter 11's classes, redefined here so this notebook stands alone. Each representation is an explicit object: token cost, preserved dimensions, and pinned facts. PIN admits FULL only.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Representation:
    level: str
    tokens: int
    dims: frozenset
    facts: tuple = ()  # (key, value) pairs pinned at this tier

@dataclass
class Episode:
    id: str
    label: str
    retention: str
    legal: tuple
    reps: dict

EPISODES = [
    Episode('rule', 'Governing constraint', 'PIN', ('FULL',),
            {'FULL': Representation('FULL', 120, frozenset({'exact', 'authority'}),
                                     (('prohibition', 'never-modify-migrations'),))}),
    Episode('adr', 'Architecture decision', 'COMPRESSIBLE', ('FULL', 'COMPACT', 'ANCHOR'),
            {'FULL': Representation('FULL', 500, frozenset({'exact', 'rationale', 'narrative', 'provenance'}),
                                     (('version', '18.2'), ('db', 'PostgreSQL'))),
             'COMPACT': Representation('COMPACT', 150, frozenset({'exact', 'rationale'}),
                                        (('version', '18.2'), ('db', 'PostgreSQL'))),
             'ANCHOR': Representation('ANCHOR', 40, frozenset({'identity'}),
                                       (('db', 'PostgreSQL'),))}),
    Episode('invg', 'Resolved investigation', 'COMPRESSIBLE', ('FULL', 'COMPACT', 'ANCHOR'),
            {'FULL': Representation('FULL', 800, frozenset({'exact', 'rationale', 'narrative', 'provenance'})),
             'COMPACT': Representation('COMPACT', 200, frozenset({'rationale', 'outcome'})),
             'ANCHOR': Representation('ANCHOR', 45, frozenset({'identity'}))}),
    Episode('fail', 'Recent compiler failure', 'COMPRESSIBLE', ('FULL', 'COMPACT'),
            {'FULL': Representation('FULL', 600, frozenset({'exact', 'narrative'})),
             'COMPACT': Representation('COMPACT', 180, frozenset({'outcome'}))}),
    Episode('log', 'Large historical log', 'COMPRESSIBLE', ('FULL', 'COMPACT', 'ANCHOR'),
            {'FULL': Representation('FULL', 1800, frozenset({'narrative', 'detail'})),
             'COMPACT': Representation('COMPACT', 300, frozenset({'outcome'})),
             'ANCHOR': Representation('ANCHOR', 50, frozenset({'identity'}))}),
]
by_ep = {e.id: e for e in EPISODES}
generation_count = len(EPISODES)
print(f'{len(EPISODES)} episodes; representations generated once ({generation_count} generation events).')

## Baseline — low pressure renders FULL everywhere

The selector may choose only representations the retention contract permits. Anything else is illegal, not merely inadvisable.

In [ ]:
def choose(episode, level):
    if level not in episode.legal:
        return ('ILLEGAL', None)
    return ('LEGAL', episode.reps[level])

for e in EPISODES:
    status, rep = choose(e, 'FULL')
    print(f"{e.label:24s} class={e.retention:12s} FULL -> {status} ({rep.tokens} tokens)")
    assert status == 'LEGAL'
status, _ = choose(by_ep['rule'], 'ANCHOR')
print(f'PIN constraint at ANCHOR -> {status}')
assert status == 'ILLEGAL'

## Intervention 1 — one item, three pressures

The architecture decision is COMPRESSIBLE at every pressure. Only the active fidelity moves. Retention never changes.

In [ ]:
adr = by_ep['adr']
for pressure, level in [(0.20, 'FULL'), (0.55, 'COMPACT'), (0.90, 'ANCHOR')]:
    status, rep = choose(adr, level)
    print(f'pressure={pressure:.2f} retention={adr.retention} active={level} ({status}, {rep.tokens} tokens)')
    assert adr.retention == 'COMPRESSIBLE'
    assert status == 'LEGAL'

## Intervention 2 — age is not permission

An old governing rule against a new verbose log. The naive age-only policy demotes the old rule and preserves the new log: exactly backwards. Retention legality first, age only as scheduling input.

In [ ]:
old_rule, new_log = by_ep['rule'], by_ep['log']
for eid, level in [(old_rule.id, 'ANCHOR'), (new_log.id, 'FULL')]:
    status, _ = choose(by_ep[eid], level)
    print(f"naive age-only: {by_ep[eid].label} -> {level} ({status})")
assert choose(old_rule, 'ANCHOR')[0] == 'ILLEGAL'
assert choose(new_log, 'ANCHOR')[0] == 'LEGAL'
print('Old does not mean low-fidelity-permitted. Retention-aware: rule FULL, log may demote.')

## Observation — cliff versus progressive over a pressure trajectory

In [ ]:
PRESSURES = [0.20, 0.45, 0.70, 0.90]

def flat_schedule(episodes, pressure):
    return {e.id: ('SUMMARY' if pressure >= 0.70 else 'FULL') for e in episodes}

def progressive_schedule(episodes, pressure):
    level = 'FULL' if pressure < 0.45 else ('COMPACT' if pressure < 0.70 else 'ANCHOR')
    return {e.id: (level if level in e.legal else 'FULL') for e in episodes}

def render_total(episodes, selection):
    return sum(120 if selection[e.id] == 'SUMMARY' else e.reps[selection[e.id]].tokens for e in episodes)

flat_levels = [flat_schedule(EPISODES, p) for p in PRESSURES]
prog_levels = [progressive_schedule(EPISODES, p) for p in PRESSURES]
print('pressure  flat-tokens  prog-tokens')
for p, f, g in zip(PRESSURES, flat_levels, prog_levels):
    print(f'{p:8.2f} {render_total(EPISODES, f):11d} {render_total(EPISODES, g):11d}')
print(f"trajectory totals: flat={sum(render_total(EPISODES, f) for f in flat_levels)}, "
      f"progressive={sum(render_total(EPISODES, g) for g in prog_levels)}")
assert all(choose(e, prog_levels[-1][e.id])[0] == 'LEGAL' for e in EPISODES)
print('Tokens and legality only: no claim about model behaviour.')

## Generate once, select many — cadence split with hysteresis

Representations were created once. The renderer selects repeatedly without regenerating anything. Separate demotion (0.80) and promotion (0.65) thresholds stop thrash when pressure oscillates around the single threshold (0.75) they replace. Plumbing only.

In [ ]:
DEMOTE, PROMOTE, SINGLE = 0.80, 0.65, 0.75
OSC = [0.74, 0.76, 0.74, 0.76, 0.82, 0.60]
cur_h, cur_s = ({e.id: 'FULL' for e in EPISODES}, {e.id: 'FULL' for e in EPISODES})
sw_h, sw_s, selections = 0, 0, 0
legal_down = [e.id for e in EPISODES if 'COMPACT' in e.legal]
for p in OSC:
    selections += 1
    for eid in legal_down:
        if p >= DEMOTE and cur_h[eid] == 'FULL':
            cur_h[eid] = 'COMPACT'
            sw_h += 1
        elif p <= PROMOTE and cur_h[eid] == 'COMPACT':
            cur_h[eid] = 'FULL'
            sw_h += 1
        want = 'COMPACT' if p >= SINGLE else 'FULL'
        if want != cur_s[eid]:
            cur_s[eid] = want
            sw_s += 1
print(f'selections={selections}, generations added=0 (total still {generation_count})')
print(f'switches with hysteresis: {sw_h}; single threshold: {sw_s}')
assert sw_h <= sw_s

## Anchor, floor, consistency, Pareto

In [ ]:
anchor = {'episode_id': 'invg', 'outcome': 'serializer ruled out', 'invariant': 'timestamps intact',
          'entity': 'checkout', 'source_ref': 'turns 12-19'}
print('anchor (identity, not prose):', anchor)
assert anchor['episode_id'] == 'invg' and 'narrative' not in anchor
print('An anchor never pretends to contain detail it lacks.')

print('fidelity floors (PROVISIONAL term): rule=FULL, adr=COMPACT_WITH_RATIONALE, invg=ANCHOR')
assert by_ep['rule'].legal == ('FULL',)

# Cross-tier consistency: shared facts must agree exactly across tiers of one episode.
for e in EPISODES:
    seen = {}
    for lvl, rep in e.reps.items():
        for k, v in rep.facts:
            assert seen.setdefault(k, v) == v, f'contradiction across tiers: {e.id}.{k}'
print('cross-tier consistency: no contradictions, no lost-and-reappearing invariants.')

# Pareto: a tier survives only with an operating point no neighbour offers.
reps = by_ep['adr'].reps
dominated = [lvl for lvl, r in reps.items()
             if any(o != lvl and reps[o].tokens <= r.tokens and set(reps[o].dims) >= set(r.dims)
                    for o in reps)]
print('dominated tiers on adr:', dominated or 'none')
assert dominated == []

## Try it

1. Demote `fail` to ANCHOR and confirm ILLEGAL: its legal range stops at COMPACT.
2. Replay the oscillation with PROMOTE = 0.79 and watch hysteresis collapse toward single-threshold thrash.
3. Add a 600-token COMPACT tier to `adr` preserving only `identity` and confirm the Pareto check flags it dominated.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(choose(by_ep['fail'], 'ANCHOR'))

## What this demonstrates

- Retention determines which transformations are legal; fidelity determines which legal representation is active now.
- Generating representations and selecting among them run on different cadences: one generation event, many selections, zero regeneration.
- Age alone is not permission to lose detail: the age-only schedule violates the governing rule.

## What this does not demonstrate

- That three fidelity tiers are optimal.
- That progressive fidelity improves real model behaviour.
- That age belongs in every schedule, or that deterministic rendering guarantees correctness.
- That anchors suffice for all old context, that any tiering formula is correct, or that fidelity selection replaces externalisation.

## Connection to the chapter

Even the lowest honest resident representations still consume window space:

> Even the lowest honest resident representations still consume window space. The next mechanism changes the question from "how much of this should stay visible?" to "does this need to remain in the live window at all?"

That is Chapter 13.